# 03 — Baseline classique : HOG + SVM

> **Projet AutoMaze AI** — classification de 6 usagers de la route, entraînée de zéro.
> Règles du cours : pas de YOLO, pas de détecteur, **pas de poids préentraînés** (`pretrained=True` interdit).
> Les cellules `TODO` sont à écrire par l'équipe. Ne supprimez pas les questions : répondez-y en dessous, elles nourrissent le rapport.

**Objectif :** obtenir rapidement un **score de référence** sur la validation. Le CNN du notebook 04 devra le battre.

**Pourquoi c'est important :**
- si la baseline est proche du hasard (1/6 ≈ 17 %), le problème vient probablement des données ;
- sans point de comparaison, un score de CNN ne veut rien dire pour le jury.

**HOG en une phrase :** on découpe l'image en petites cellules et on compte, dans chacune, les directions des contours.
Le SVM apprend ensuite une frontière entre les classes dans cet espace de descripteurs.

## 0. Imports

In [ ]:
import time

import pandas as pd
import matplotlib.pyplot as plt
import cv2
from sklearn.svm import LinearSVC, SVC                 # classifieurs SVM (linéaire = rapide, RBF = plus souple)
from sklearn.preprocessing import StandardScaler       # mettre les descripteurs à la même échelle
from sklearn.pipeline import make_pipeline             # enchaîner normalisation + classifieur proprement
from sklearn.dummy import DummyClassifier              # score plancher (hasard / classe majoritaire)
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay

In [ ]:
# --- Configuration commune (identique dans chaque notebook) ---
from pathlib import Path
import random

import numpy as np

# Exécution sur Google Colab ou en local : le même notebook fonctionne dans les deux cas.
try:
    import google.colab  # noqa: F401  (ce module n'existe que sur Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Sur Colab, tout ce qui est écrit dans /content disparaît à la fin de la session.
    # On travaille donc dans Google Drive : données, figures et poids y sont conservés.
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/AutoMaze_Ai")  # TODO(équipe) : adapter au dossier partagé de l'équipe
else:
    # En local, ce notebook est dans notebooks/ : la racine du projet est le dossier parent.
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Colab" if IN_COLAB else "Local", "| racine du projet :", ROOT)
DATA_DIR = ROOT / "data"          # données locales, ignorées par Git
RAW_DIR = DATA_DIR / "raw"        # fichiers téléchargés tels quels (ex. annotations COCO)
CROPS_DIR = DATA_DIR / "crops"    # recadrages produits par le notebook 02
MODELS_DIR = ROOT / "models"      # poids + métadonnées (les .pt sont ignorés par Git)
FIGURES_DIR = ROOT / "reports" / "figures"  # figures à reprendre dans le rapport

for d in (RAW_DIR, CROPS_DIR, MODELS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Ordre des classes = source de vérité pour TOUT le projet (notebooks + simulateur).
# Ne le changez plus une fois l'entraînement commencé : l'indice 0 doit toujours désigner la même classe.
CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]
CLASSES_FR = ["Personne", "Vélo", "Voiture", "Moto", "Bus", "Camion"]

# Graine aléatoire : rend les résultats reproductibles (même découpage, mêmes initialisations).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1. Charger les recadrages train et validation

On **n'utilise pas** `data/crops/test` ici.

In [ ]:
# TODO(équipe) : fonction load_split(split) -> (X_images, y) qui lit data/crops/<split>/<classe>/*.jpg
# y = indice de la classe dans CLASSES (respecter cet ordre !).

## 2. Extraire les descripteurs HOG

`cv2.HOGDescriptor` suffit (pas besoin d'installer scikit-image). Paramètres à comprendre et à faire varier :
taille de fenêtre, de bloc, de cellule et nombre d'orientations.

**Question :** HOG utilise-t-il la couleur ? Qu'est-ce que cela implique pour distinguer un bus d'un camion ?

In [ ]:
# TODO(équipe) : créer un HOGDescriptor adapté à INPUT_SIZE, calculer les descripteurs de train et de val.
# Afficher la dimension d'un descripteur.

## 3. Score plancher

In [ ]:
# TODO(équipe) : DummyClassifier(strategy="most_frequent") puis "stratified" : F1 macro sur la validation.

## 4. Entraîner le SVM

Commencez par `LinearSVC`, puis essayez `SVC(kernel="rbf")` si le temps le permet. Notez chaque essai dans le journal des runs du rapport.

In [ ]:
# TODO(équipe) : make_pipeline(StandardScaler(), LinearSVC(C=...)), entraînement, mesure du temps.
# Faire varier C (ex. 0.01, 0.1, 1, 10) et garder le meilleur selon le F1 macro de VALIDATION.

## 5. Analyse

In [ ]:
# TODO(équipe) : classification_report et matrice de confusion (normalisée par ligne) sur la validation.
# Sauvegarder la figure dans FIGURES_DIR.

## 6. Conclusions

- Meilleur F1 macro de validation : …
- Classes les mieux / les moins bien reconnues : …
- Confusions principales et hypothèse d'explication : …

Ces résultats vont dans la ligne « Baseline » du journal des runs (section 4 du rapport).